# Texas SBIR/STTR Impact — Guided Analysis
## Step 01 · Meet the data

This is a **slow, explained** walk-through. Every step answers four questions:

- **Why** — the question this step helps answer
- **Who** — where the data comes from and who produced it
- **How** — what the code actually does
- **What it means** — how to read the result

**Goal of Step 01:** load the raw SBIR/STTR award data, understand *what one row is*, then narrow
it to our study population — **every award that was *active* in Texas during 2016–2025**.

### 1. Set up our tools and reach the data
**Why:** we need Python's data tools (pandas) and access to the data file in your Google Drive.
**How:** mount Drive and import pandas (tables) and `re` (a small text helper).

In [ ]:
import pandas as pd   # pandas = spreadsheets in code (tables called 'DataFrames')
import re              # for cleaning up company-name text later
from google.colab import drive
drive.mount('/content/drive')

FOLDER = '/content/drive/MyDrive/fast_datasets/SBIR_TX_Guided_Analysis/'
AWARD_FILE = FOLDER + 'awards_search.csv'
import os
if not os.path.exists(AWARD_FILE):
    AWARD_FILE = '/content/drive/MyDrive/fast_datasets/awards_search_1780524328.csv'
print('Reading award data from:', AWARD_FILE)

### 2. Who made this data, and what is it?
**Who:** the **U.S. Small Business Administration (SBA)** publishes this on **SBIR.gov** — the
official record of every **SBIR/STTR award**, pulled from all 11 agencies (Defense, NIH, NASA,
NSF, Energy, etc.).

**Why it's our foundation:** the question *'what is the impact of SBIR/STTR in Texas?'* starts from
*who got awards*. Everything else (still alive? won contracts? where?) attaches to **these** rows.

**How:** `pd.read_csv` loads the file into a table `df`; we print its size and columns.

In [ ]:
df = pd.read_csv(AWARD_FILE, low_memory=False)
print('Rows (awards):', f'{len(df):,}')
print('Columns:', df.shape[1])
print()
for c in df.columns:
    print('  -', c)

### 3. What is *one row*?
**Key idea:** **one row = one award action** to one company. A company can appear on many rows.
The columns we lean on most:

| Column | What it tells us |
|---|---|
| `Company Name` | who received it |
| `Agency` / `Branch` | which agency funded it (DoD, HHS/NIH, NASA…) |
| `Program` | SBIR or STTR |
| `Phase` | Phase I (feasibility) or Phase II (build-out) |
| `Award Amount` | dollars for that award |
| `Award Year` | when it was **awarded** (work starts) |
| `Contract End Date` | when the work is scheduled to **finish** |
| `State` / `City` / `ZIP` | where the company is |
| `UEI` | the government's unique company ID (better than the name for matching) |

Notice we have **both** a start (`Award Year`) and an end (`Contract End Date`) — that matters in
the next step. **How:** print one real award top-to-bottom.

In [ ]:
example = df.iloc[0]
for field in ['Company Name','Agency','Program','Phase','Award Amount','Award Year','Contract End Date','City','State','UEI']:
    print(f'{field:18}: {example[field]}')

### 4. Narrow to our study population: **active in Texas, 2016–2025**
**Why (this is the important part):** we want **all award *activity* that happened in 2016–2025** —
not just awards *started* then. An award funds work over a **period**: from its `Award Year` to its
`Contract End Date`. A company that won in **2014** but whose contract ran until **2016** was still
*doing the work* in our window, so it belongs in the study. If we filtered only on *award year* we'd
wrongly drop those companies.

**How — the “overlap” rule:** keep an award if its work period **touches** 2016–2025, i.e.

- it **started on or before** the end of 2025 (`Award Year` ≤ 2025), **and**
- it **ended on or after** the start of 2016 (`Contract End Date` ≥ Jan 1 2016).

If an award has **no end date**, we fall back to its award year (so it's included only if it was made
in the window — a conservative choice we'll state whenever it matters).

In [ ]:
# Parse the start and end of each award's work period
df['year']     = pd.to_numeric(df['Award Year'], errors='coerce')
df['award_dt'] = pd.to_datetime(df['Proposal Award Date'], errors='coerce')
df['end_dt']   = pd.to_datetime(df['Contract End Date'], errors='coerce')

WIN_START = pd.Timestamp('2016-01-01')
WIN_END   = pd.Timestamp('2025-12-31')

# start = actual award date, or Jan 1 of the award year if the date is missing
start = df['award_dt'].fillna(pd.to_datetime(df['year'], format='%Y', errors='coerce'))
# end = contract end date, or fall back to the start (i.e., no known end -> judge by award year)
end   = df['end_dt'].fillna(start)

# 'active in the window' = the work period overlaps 2016-2025
active = (start <= WIN_END) & (end >= WIN_START)
tx = df[(df['State'] == 'TX') & active].copy()

# Cleaned company key so different spellings of one firm group together
def normalize(name):
    if pd.isna(name): return ''
    s = str(name).upper().strip()
    s = re.sub(r'[.,&\-/]', ' ', s)
    s = re.sub(r'\b(INC|LLC|LP|LTD|CORP|CORPORATION|CO|COMPANY|INCORPORATED)\b', '', s)
    return re.sub(r'\s+', ' ', s).strip()
tx['name_key'] = tx['Company Name'].apply(normalize)

print('Texas awards ACTIVE in 2016-2025:', f'{len(tx):,}')
print('Unique companies               :', f"{tx['name_key'].nunique():,}")
print('Total award dollars            : $', f"{pd.to_numeric(tx['Award Amount'], errors='coerce').sum():,.0f}", sep='')

In [ ]:
# See what the overlap rule added vs. an 'award-year-only' filter
made_in_window = df[(df['State']=='TX') & df['year'].between(2016,2025)]
added = tx[~tx.index.isin(made_in_window.index)]
print(f'Award-year-only filter : {len(made_in_window):,} awards')
print(f'Active-in-window filter: {len(tx):,} awards  (+{len(added)} added)')
print('\nThe added awards were MADE in these years but stayed active into the window:')
print(added['year'].value_counts().sort_index().to_string())

### ✅ What this means
Our study population is **every SBIR/STTR award active in Texas during 2016–2025**:

- **~3,612 awards** to **~916 unique companies**, worth about **$1.97 billion**.
- That's **+218 awards and +39 companies** vs. an award-year-only filter — mostly firms that **won in
  2014–2015 but whose contracts ran into 2016**. They were genuinely active in our window, so keeping
  them makes the picture honest.

**One nuance to remember (we'll use it later):** *being active in the window* is how we choose **who's
in the study**. But when we later chart **awards made per year**, we'll use the award year for that
specific chart — otherwise a 2014 award would show up as 2016 activity. Same data, different question.

Next up — *Step 02: **Who funds them?*** which agencies back these Texas companies, and how the mix
has shifted over the decade.

> **Check your understanding:** in the last cell, which single year contributed the most *added* awards,
> and why does that make sense given contracts typically run 1–2 years?